In [2]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
from yellowbrick.cluster import KElbowVisualizer
import matplotlib.pyplot as plt
import pandas as pd 
import seaborn as sns
from sklearn.cluster import KMeans
from sksurv.base import SurvivalAnalysisMixin as s
from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score
from sksurv.preprocessing import encode_categorical
from sksurv.datasets import load_gbsg2
from sksurv.functions import StepFunction
from sksurv.linear_model import CoxPHSurvivalAnalysis, CoxnetSurvivalAnalysis
from sksurv.ensemble import (ComponentwiseGradientBoostingSurvivalAnalysis, 
                            RandomSurvivalForest, 
                            ExtraSurvivalTrees, 
                            GradientBoostingSurvivalAnalysis, 
                            ExtraSurvivalTrees)
from sksurv.meta import EnsembleSelection, EnsembleSelectionRegressor
from sksurv.metrics import integrated_brier_score
from matplotlib.colors import ListedColormap
from mlxtend.evaluate import paired_ttest_5x2cv
from mlxtend.evaluate import combined_ftest_5x2cv
from lifelines import KaplanMeierFitter
from scipy.cluster import hierarchy
from lifelines.statistics import logrank_test, multivariate_logrank_test, pairwise_logrank_test
from sklearn import preprocessing
from sklearn.model_selection import StratifiedKFold, KFold
from lifelines.plotting import add_at_risk_counts
import scipy.stats
import sklearn
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import optuna
from sklearn.model_selection import cross_val_score
from sksurv.metrics import integrated_brier_score
from lifelines import CoxPHFitter
from lifelines.statistics import proportional_hazard_test
import scipy.stats as stats
from statsmodels.stats.outliers_influence import variance_inflation_factor 
 
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'
from sklearn.preprocessing import MinMaxScaler

In [3]:
# OUS: Train data
OUS_D1 = pd.read_csv('OUS_D1.csv')
OUS_D2 = pd.read_csv('OUS_D2.csv')
OUS_D3 = pd.read_csv('OUS_D3.csv')
OUS_DFS_target = pd.read_csv('OUS_DFS_target.csv')
OUS_OS_target = pd.read_csv('OUS_OS_target.csv')
response_OUS = pd.read_csv('response_ous.csv', sep=';')

# MAASTRO: Test data 
MAASTRO_D1 = pd.read_csv('MAASTRO_D1.csv')
MAASTRO_D2 = pd.read_csv('MAASTRO_D2.csv')
MAASTRO_D3 = pd.read_csv('MAASTRO_D3.csv')
MAASTRO_DFS_target = pd.read_csv('MAASTRO_DFS_target.csv')
MAASTRO_OS_target = pd.read_csv('MAASTRO_OS_target.csv')
response_MAASTRO = pd.read_csv('maastro_response_full.csv', sep=',')

In [4]:
# Need to choose patient_id from OUS_D1 in response_OUS
data = list(OUS_D1['patient_id'])
mask = response_OUS['patient_id'].isin(data)
response_OUS = response_OUS[mask] 

# Merge OUS_D2 with response_OUS
clinical_train = pd.merge(OUS_D1, response_OUS, on='patient_id', how='inner')
clinical_train = clinical_train.loc[:, ~clinical_train.columns.isin(['DFS', 'event_DFS', 'LRC', 'event_LRC'])]

In [5]:
# Drop patient_id column
clinical_train = clinical_train.drop('patient_id', axis=1)

In [6]:
# Check null values in D1
clinical_train.isnull().sum().sum()

0

## Test dataset: MAASTRO 

In [7]:
(MAASTRO_D1['patient_id'] == MAASTRO_OS_target['patient_id']).sum()

99

In [8]:
# Rename the column name of response_MAASTRO 
response_MAASTRO.rename(columns = {'Index' : 'patient_id'}, inplace = True)

In [9]:
# need to choose patient_id from MAASTRO_D1 in response_MAASTRO
data = list(MAASTRO_D1['patient_id'])
mask = response_MAASTRO['patient_id'].isin(data)
response_MAASTRO = response_MAASTRO[mask] 

In [10]:
# Merge MAASTRO_D2 with response_MAASTRO
clinical_test = pd.merge(MAASTRO_D1, response_MAASTRO, on='patient_id', how='inner')
clinical_test = clinical_test.loc[:, ~clinical_test.columns.isin(['DFS', 'DFS_event', 'LRC', 'LRC_event'])]

In [11]:
# Drop patient_id column
clinical_test = clinical_test.drop('patient_id', axis=1)

In [12]:
# Check if some rows have null values in OS, OS_event -> Remove those rows
clinical_test[clinical_test.isnull().any(axis=1)]
clinical_test = clinical_test.dropna(how='any',axis=0) 

,age,female,cavum_oris,oropharynx,hypopharynx,larynx,histgrade_high,hpv_related,charlson,pack_years,uicc8_III-IV,SUVpeak,MTV,TLG,OS,OS_event


In [13]:
# X
X = clinical_train.loc[:, ~clinical_train.columns.isin(['OS', 'event_OS'])]

# y 
y = clinical_train.loc[:, ['OS', 'event_OS']]

In [14]:
# Set lower, upper time point and gbsg_times for IBS calculation later 

# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

# Shape
print('X_train: ', X.shape)
print('y_train: ', y.shape)

clinical_test.rename(columns = {'OS_event' : 'event_OS'}, inplace = True)

# X
X_MAASTRO = clinical_test.loc[:, ~clinical_test.columns.isin(['OS', 'event_OS'])]

# y y_MAASTRO
y_MAASTRO = clinical_test.loc[:, ['OS', 'event_OS']]
lower, upper = np.percentile(y_MAASTRO['OS'], [10, 90])
gbsg_times = np.arange(lower, upper)

# y into array 
lists = [] 
for i, j in zip(y_MAASTRO['event_OS'], y_MAASTRO['OS']): 
    lists.append((i, j))

y_MAASTRO = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

clinical_test.shape

X_train:  (139, 14)
y_train:  (139,)


(99, 16)

# Feature Selection: PLSR

In [15]:
selected_features = ["hpv_related",
"MTV",
"TLG"]

# Selecting features in the DataFrame
X_plsr = X[selected_features]
X_new = X_plsr.copy()

In [16]:
X_MAASTRO_plsr = X_MAASTRO[selected_features]
MAASTRO_new = X_MAASTRO_plsr.copy()

# Standardization

In [17]:
# Standardize X_new, the new data with the selected features only 
categorical_columns = ['female', 
                        'cavum_oris',
                        'oropharynx',
                        'hypopharynx',
                        'larynx',
                        'histgrade_high',
                        'hpv_related',
                        'charlson',
                        'uicc8_III-IV']

# Set the columns_to_drop which are categorical 
columns_to_drop = [col for col in X_new.columns if col in categorical_columns]

# Drop the columns if they exist in the DataFrame and save the numeric part in X_new_numeric
X_new_categoric = X_new[columns_to_drop]
X_new_numeric = X_new.drop(columns=columns_to_drop, inplace=False)

# Do the standardization for the numeric part 
scaler = MinMaxScaler() 
X_new_numeric_columns = X_new_numeric.columns
X_new_numeric_index = X_new_numeric.index 
X_new_numeric_std = scaler.fit_transform(X_new_numeric)
X_new_numeric_std = pd.DataFrame(X_new_numeric_std,
                                 columns=X_new_numeric_columns, 
                                 index=X_new_numeric_index)
X_new_std = pd.concat([X_new_numeric_std, X_new_categoric], axis=1)

# Change the order of the X_new_std 
X_new_std = X_new_std[X_new.columns]

In [26]:
# Standardize X_MAASTRO 
# Set the columns_to_drop which are categorical 
columns_to_drop = [col for col in MAASTRO_new.columns if col in categorical_columns]

# Drop the columns if they exist in the DataFrame and save the numeric part in X_new_numeric
MAASTRO_new_categoric = MAASTRO_new[columns_to_drop]
MAASTRO_new_numeric = MAASTRO_new.drop(columns=columns_to_drop, inplace=False)

# Do the standardization for the numeric part 
MAASTRO_new_numeric_columns = MAASTRO_new_numeric.columns
MAASTRO_new_numeric_index = MAASTRO_new_numeric.index 
MAASTRO_new_numeric_std = scaler.transform(MAASTRO_new_numeric)
MAASTRO_new_numeric_std = pd.DataFrame(MAASTRO_new_numeric_std,
                                 columns=MAASTRO_new_numeric_columns, 
                                 index=MAASTRO_new_numeric_index)
MAASTRO_new_std = pd.concat([MAASTRO_new_numeric_std, MAASTRO_new_categoric], axis=1)

# Change the order of the X_new_std 
MAASTRO_new = MAASTRO_new[X_new.columns]
MAASTRO_new_std = MAASTRO_new_std[MAASTRO_new.columns]

In [27]:
X_new

,hpv_related,MTV,TLG
0,0.0,7.934,86.228420
1,0.0,1.656,7.040100
2,0.0,14.502,83.569669
3,0.0,2.440,5.567091
4,0.0,3.668,16.150550
...,...,...,...
134,1.0,3.650,26.280140
135,1.0,18.967,101.754834
136,1.0,6.370,66.273201
137,1.0,12.443,71.832443


In [28]:
X_new_std

,hpv_related,MTV,TLG
0,0.0,0.074159,0.044798
1,0.0,0.008512,0.001900
2,0.0,0.142839,0.043358
3,0.0,0.016710,0.001102
4,0.0,0.029551,0.006835
...,...,...,...
134,1.0,0.029363,0.012323
135,1.0,0.189529,0.053209
136,1.0,0.057805,0.033988
137,1.0,0.121309,0.037000


In [29]:
MAASTRO_new

,hpv_related,MTV,TLG
0,1,22.841,263.611623
1,0,5.660,36.980700
2,0,7.791,74.636342
3,0,7.908,46.791979
4,1,15.237,107.637514
...,...,...,...
94,0,6.110,144.490782
95,0,7.182,69.214868
96,1,16.483,102.594274
97,1,9.981,103.229492


In [30]:
MAASTRO_new_std

,hpv_related,MTV,TLG
0,1,0.230038,0.140892
1,0,0.050381,0.018119
2,0,0.072664,0.038519
3,0,0.073887,0.023434
4,1,0.150525,0.056396
...,...,...,...
94,0,0.055086,0.076361
95,0,0.066296,0.035582
96,1,0.163554,0.053664
97,1,0.095564,0.054008


# Modelling 

### 1. CoxPHSurvivalAnalysis

#### Train

In [31]:
# Setting the y format for skf below  
y = clinical_train[['OS', 'event_OS']]

# Running to optuna for hyperparameter tuning 
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class()
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Min Max Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = MinMaxScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                gbsg_times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(gbsg_times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, gbsg_times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxPHSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxPHSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-13 17:55:26,311] A new study created in memory with name: no-name-91dc8cfb-15fd-4cc6-9bd9-08bda29ec3d9


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.6623376623376623
Fold 2 C-index: 0.7678571428571429
Fold 3 C-index: 0.803921568627451
Fold 4 C-index: 0.8270042194092827


[I 2024-04-13 17:55:27,441] A new study created in memory with name: no-name-27d47374-d5e7-449d-ad44-4fbe50fd1b9f


Fold 5 C-index: 0.6197183098591549
[I 2024-04-13 17:55:27,428] Trial 0 finished with value: 0.7361677806181388 and parameters: {}. Best is trial 0 with value: 0.7361677806181388.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.7361677806181388], datetime_start=datetime.datetime(2024, 4, 13, 17, 55, 26, 455302), datetime_complete=datetime.datetime(2024, 4, 13, 17, 55, 27, 428052), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.7361677806181388


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.1791590650279046
Fold 2 IBS: 0.20170223404947618
Fold 3 IBS: 0.1750546844970336
Fold 4 IBS: 0.14253965894637302
Fold 5 IBS: 0.2695925587123099
[I 2024-04-13 17:55:28,621] Trial 0 finished with value: 0.19360964024661945 and parameters: {}. Best is trial 0 with value: 0.19360964024661945.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.19360964024661945], datetime_start=datetime.datetime(2024, 4, 13, 17, 55, 27, 478579), datetime_complete=datetime.datetime(2024, 4, 13, 17, 55, 28, 620394), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.19360964024661945


In [32]:
# Setting a dictionary to save the train results 
train_cindex = {} 
train_ibs = {} 

# Saving the values to the dictionary 
train_cindex['CoxPH'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxPH'] = np.round(study_ibs.best_value, 3)

In [33]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.736
train_ibs:  0.194


#### Test

In [34]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [35]:
# Test on MAASTRO 
cph = CoxPHSurvivalAnalysis()

cph.fit(X_new_std, y)

# Save C-index 
c_index = cph.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print('Concordance index:', c_index)

# Save IBS 
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
gbsg_times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(gbsg_times) for fn in cph.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, gbsg_times)
ibs = np.round(ibs, 3)
print('IBS score:', ibs)

CoxPHSurvivalAnalysis()

Concordance index: 0.623
IBS score: 0.234


In [36]:
# Setting a dictionary to save the test results 
test_cindex = {} 
test_ibs = {} 

In [37]:
# Saving the values to the dictionary 
test_cindex['CoxPH'] = c_index
test_ibs['CoxPH'] = ibs

### 2. CoxnetSurvivalAnalysis - Ridge 

#### Train

In [38]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

# Running to optuna for hyperparameter tuning 
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class(l1_ratio=0.0000001, 
                            fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Min Max Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = MinMaxScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                gbsg_times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(gbsg_times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, gbsg_times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-13 17:55:29,211] A new study created in memory with name: no-name-1c943c8b-8e21-454e-b854-e2c2df70cadd


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.7575757575757576
Fold 2 C-index: 0.6004464285714286
Fold 3 C-index: 0.7328431372549019


[I 2024-04-13 17:55:29,942] A new study created in memory with name: no-name-103bdd14-b9f0-4cf6-b0c7-7b34211b684f


Fold 4 C-index: 0.810126582278481
Fold 5 C-index: 0.5868544600938967
[I 2024-04-13 17:55:29,932] Trial 0 finished with value: 0.6975692731548933 and parameters: {}. Best is trial 0 with value: 0.6975692731548933.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.6975692731548933], datetime_start=datetime.datetime(2024, 4, 13, 17, 55, 29, 251179), datetime_complete=datetime.datetime(2024, 4, 13, 17, 55, 29, 919702), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.6975692731548933


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.21397651764739065
Fold 2 IBS: 0.221577916505828
Fold 3 IBS: 0.20453594517659848
Fold 4 IBS: 0.2247380366950916
Fold 5 IBS: 0.21812431575089974
[I 2024-04-13 17:55:30,688] Trial 0 finished with value: 0.2165905463551617 and parameters: {}. Best is trial 0 with value: 0.2165905463551617.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.2165905463551617], datetime_start=datetime.datetime(2024, 4, 13, 17, 55, 30, 105254), datetime_complete=datetime.datetime(2024, 4, 13, 17, 55, 30, 688669), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.2165905463551617


In [39]:
train_cindex['CoxRidge'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxRidge'] = np.round(study_ibs.best_value, 3)

In [40]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.698
train_ibs:  0.217


#### Test

In [41]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [42]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params['l1_ratio'] = 0.0000001
    best_params['fit_baseline_model']=True
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
gbsg_times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(gbsg_times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, gbsg_times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1e-07)

test_cindex : 0.589


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1e-07)

test_ibs:  0.221


In [43]:
# Saving the values to the dictionary 
test_cindex['CoxRidge'] = c_index
test_ibs['CoxRidge'] = ibs

### 3. CoxnetSurvivalAnalysis - Lasso

#### Train

In [44]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Create and fit survival model 
        model = model_class(l1_ratio=1, 
                            fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Min Max Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = MinMaxScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                gbsg_times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(gbsg_times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, gbsg_times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=1, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-13 17:55:33,374] A new study created in memory with name: no-name-bd668467-10f7-4baf-924e-49c189ab9722


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 C-index: 0.6623376623376623
Fold 2 C-index: 0.7678571428571429
Fold 3 C-index: 0.7990196078431373
Fold 4 C-index: 0.8270042194092827


[I 2024-04-13 17:55:35,622] A new study created in memory with name: no-name-b49c04b1-2d3f-464d-b448-d3b0053945c1


Fold 5 C-index: 0.6244131455399061
[I 2024-04-13 17:55:35,581] Trial 0 finished with value: 0.7361263555974262 and parameters: {}. Best is trial 0 with value: 0.7361263555974262.


* Best trial for C-index: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.7361263555974262], datetime_start=datetime.datetime(2024, 4, 13, 17, 55, 33, 610049), datetime_complete=datetime.datetime(2024, 4, 13, 17, 55, 35, 580772), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for C-index: 
 0.7361263555974262


  0%|          | 0/1 [00:00<?, ?it/s]

Fold 1 IBS: 0.1790010227435467
Fold 2 IBS: 0.20300340736300934
Fold 3 IBS: 0.17476312604505478
Fold 4 IBS: 0.1424966344952737
Fold 5 IBS: 0.2691552661970626
[I 2024-04-13 17:55:37,838] Trial 0 finished with value: 0.1936838913687894 and parameters: {}. Best is trial 0 with value: 0.1936838913687894.


* Best trial for IBS: 
 FrozenTrial(number=0, state=TrialState.COMPLETE, values=[0.1936838913687894], datetime_start=datetime.datetime(2024, 4, 13, 17, 55, 35, 788323), datetime_complete=datetime.datetime(2024, 4, 13, 17, 55, 37, 837771), params={}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={}, trial_id=0, value=None)


* Best Score for IBS: 
 0.1936838913687894


In [45]:
train_cindex['CoxLasso'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxLasso'] = np.round(study_ibs.best_value, 3)

In [46]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.736
train_ibs:  0.194


#### Test 

In [47]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [48]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params['l1_ratio'] = 1
    best_params['fit_baseline_model']=True
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
gbsg_times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(gbsg_times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, gbsg_times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1)

test_cindex : 0.624


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=1)

test_ibs:  0.232


In [49]:
# Saving the values to the dictionary 
test_cindex['CoxLasso'] = c_index
test_ibs['CoxLasso'] = ibs

### 4. CoxnetSurvivalAnalysis - ElasticNet

#### Train

In [50]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        l1_ratio = trial.suggest_float("l1_ratio", 0.0001, 1)
        
        # Create and fit survival model 
        model = model_class(l1_ratio=l1_ratio, 
                           fit_baseline_model=True)
        
        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Min Max Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = MinMaxScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                gbsg_times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(gbsg_times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, gbsg_times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(CoxnetSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(CoxnetSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-13 17:55:39,208] A new study created in memory with name: no-name-b54b262b-0bff-4144-9fc8-496b732e478a


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.6623376623376623
Fold 2 C-index: 0.7678571428571429
Fold 3 C-index: 0.7990196078431373
Fold 4 C-index: 0.8312236286919831
Fold 5 C-index: 0.6244131455399061
[I 2024-04-13 17:55:41,099] Trial 0 finished with value: 0.7369702374539663 and parameters: {'l1_ratio': 0.6964995386793018}. Best is trial 0 with value: 0.7369702374539663.
Fold 1 C-index: 0.6666666666666666
Fold 2 C-index: 0.7678571428571429
Fold 3 C-index: 0.7990196078431373
Fold 4 C-index: 0.8270042194092827
Fold 5 C-index: 0.6244131455399061
[I 2024-04-13 17:55:43,206] Trial 1 finished with value: 0.7369921564632271 and parameters: {'l1_ratio': 0.28621072101688444}. Best is trial 1 with value: 0.7369921564632271.
Fold 1 C-index: 0.6666666666666666
Fold 2 C-index: 0.7678571428571429
Fold 3 C-index: 0.7990196078431373
Fold 4 C-index: 0.8270042194092827
Fold 5 C-index: 0.6244131455399061
[I 2024-04-13 17:55:45,300] Trial 2 finished with value: 0.7369921564632271 and parameters: {'l1_ratio': 0.22692876841884668}.

Fold 1 C-index: 0.6666666666666666
Fold 2 C-index: 0.7678571428571429
Fold 3 C-index: 0.7990196078431373
Fold 4 C-index: 0.8312236286919831
Fold 5 C-index: 0.6244131455399061
[I 2024-04-13 17:56:26,016] Trial 24 finished with value: 0.7378360383197672 and parameters: {'l1_ratio': 0.37291524271560916}. Best is trial 14 with value: 0.7379311235993773.
Fold 1 C-index: 0.6666666666666666
Fold 2 C-index: 0.7678571428571429
Fold 3 C-index: 0.7990196078431373
Fold 4 C-index: 0.8312236286919831
Fold 5 C-index: 0.6244131455399061
[I 2024-04-13 17:56:28,337] Trial 25 finished with value: 0.7378360383197672 and parameters: {'l1_ratio': 0.4236492178393837}. Best is trial 14 with value: 0.7379311235993773.
Fold 1 C-index: 0.6666666666666666
Fold 2 C-index: 0.7678571428571429
Fold 3 C-index: 0.7990196078431373
Fold 4 C-index: 0.8270042194092827
Fold 5 C-index: 0.6244131455399061
[I 2024-04-13 17:56:31,181] Trial 26 finished with value: 0.7369921564632271 and parameters: {'l1_ratio': 0.14172431544585

Fold 5 C-index: 0.6244131455399061
[I 2024-04-13 17:57:12,301] Trial 47 finished with value: 0.7365482965256962 and parameters: {'l1_ratio': 0.573840629919339}. Best is trial 14 with value: 0.7379311235993773.
Fold 1 C-index: 0.6666666666666666
Fold 2 C-index: 0.7678571428571429
Fold 3 C-index: 0.7990196078431373
Fold 4 C-index: 0.8312236286919831
Fold 5 C-index: 0.6244131455399061
[I 2024-04-13 17:57:13,808] Trial 48 finished with value: 0.7378360383197672 and parameters: {'l1_ratio': 0.3951993011558007}. Best is trial 14 with value: 0.7379311235993773.
Fold 1 C-index: 0.6666666666666666
Fold 2 C-index: 0.7678571428571429
Fold 3 C-index: 0.7990196078431373
Fold 4 C-index: 0.8312236286919831
Fold 5 C-index: 0.6244131455399061
[I 2024-04-13 17:57:15,511] Trial 49 finished with value: 0.7378360383197672 and parameters: {'l1_ratio': 0.34713398312216626}. Best is trial 14 with value: 0.7379311235993773.
Fold 1 C-index: 0.6666666666666666
Fold 2 C-index: 0.7678571428571429
Fold 3 C-index: 0

...

  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 IBS: 0.17901608673122438
Fold 2 IBS: 0.2032298388767962
Fold 3 IBS: 0.1748061167355499
Fold 4 IBS: 0.1424598127912613
Fold 5 IBS: 0.26925610304190817
[I 2024-04-13 17:58:29,830] Trial 0 finished with value: 0.193753591635348 and parameters: {'l1_ratio': 0.6964995386793018}. Best is trial 0 with value: 0.193753591635348.
Fold 1 IBS: 0.17902658310162128
Fold 2 IBS: 0.20334768738254913
Fold 3 IBS: 0.1748247416286342
Fold 4 IBS: 0.14244653975493216
Fold 5 IBS: 0.2692995422000311
[I 2024-04-13 17:58:31,945] Trial 1 finished with value: 0.19378901881355354 and parameters: {'l1_ratio': 0.28621072101688444}. Best is trial 0 with value: 0.193753591635348.
Fold 1 IBS: 0.17902862187768215
Fold 2 IBS: 0.20335940307073755
Fold 3 IBS: 0.1748246839697923
Fold 4 IBS: 0.1424447651679779
Fold 5 IBS: 0.26930098210916037
[I 2024-04-13 17:58:33,676] Trial 2 finished with value: 0.19379169123907009 and parameters: {'l1_ratio': 0.22692876841884668}. Best is trial 0 with value: 0.193753591635348.
Fold 

Fold 1 IBS: 0.17900709032225995
Fold 2 IBS: 0.2030657494491206
Fold 3 IBS: 0.1747902900636781
Fold 4 IBS: 0.14247976499059278
Fold 5 IBS: 0.2691789681056567
[I 2024-04-13 17:59:25,160] Trial 25 finished with value: 0.19370437258626164 and parameters: {'l1_ratio': 0.9106143729430608}. Best is trial 11 with value: 0.1936903608589698.
Fold 1 IBS: 0.1790215114696707
Fold 2 IBS: 0.20325977532303732
Fold 3 IBS: 0.17480606376688093
Fold 4 IBS: 0.14245484304626913
Fold 5 IBS: 0.26926025679236704
[I 2024-04-13 17:59:27,313] Trial 26 finished with value: 0.19376049007964502 and parameters: {'l1_ratio': 0.6273790955054189}. Best is trial 11 with value: 0.1936903608589698.
Fold 1 IBS: 0.17901879899676987
Fold 2 IBS: 0.20326597204973842
Fold 3 IBS: 0.17479803597320426
Fold 4 IBS: 0.14246228618752899
Fold 5 IBS: 0.2692403180909053
[I 2024-04-13 17:59:28,972] Trial 27 finished with value: 0.19375708225962934 and parameters: {'l1_ratio': 0.7517964642342185}. Best is trial 11 with value: 0.193690360858

Fold 1 IBS: 0.1790305164426132
Fold 2 IBS: 0.20440923769808736
Fold 3 IBS: 0.1747867033327353
Fold 4 IBS: 0.22070687724624155
Fold 5 IBS: 0.2692269146374073
[I 2024-04-13 18:00:12,106] Trial 50 finished with value: 0.20963204987141695 and parameters: {'l1_ratio': 0.08888770157152942}. Best is trial 47 with value: 0.19368905329951516.
Fold 1 IBS: 0.1790106951054058
Fold 2 IBS: 0.2031014457237615
Fold 3 IBS: 0.17477236503681473
Fold 4 IBS: 0.14249000151711563
Fold 5 IBS: 0.26913602677621024
[I 2024-04-13 18:00:13,764] Trial 51 finished with value: 0.19370210683186156 and parameters: {'l1_ratio': 0.9812035738211817}. Best is trial 47 with value: 0.19368905329951516.
Fold 1 IBS: 0.17901486574638428
Fold 2 IBS: 0.20318725247970823
Fold 3 IBS: 0.1747964977912837
Fold 4 IBS: 0.14247810829404545
Fold 5 IBS: 0.2691926014773262
[I 2024-04-13 18:00:15,887] Trial 52 finished with value: 0.1937338651577496 and parameters: {'l1_ratio': 0.8872023576102256}. Best is trial 47 with value: 0.193689053299

Fold 1 IBS: 0.17901006774965633
Fold 2 IBS: 0.20314916802846342
Fold 3 IBS: 0.17477624324192376
Fold 4 IBS: 0.14248358560184746
Fold 5 IBS: 0.26918920997380735
[I 2024-04-13 18:01:05,591] Trial 75 finished with value: 0.19372165491913967 and parameters: {'l1_ratio': 0.92219738554081}. Best is trial 47 with value: 0.19368905329951516.
Fold 1 IBS: 0.1790067916698692
Fold 2 IBS: 0.2030595718618379
Fold 3 IBS: 0.17478333856840325
Fold 4 IBS: 0.14248821506689177
Fold 5 IBS: 0.26915792666645977
[I 2024-04-13 18:01:07,196] Trial 76 finished with value: 0.19369916876669238 and parameters: {'l1_ratio': 0.9650656605801265}. Best is trial 47 with value: 0.19368905329951516.
Fold 1 IBS: 0.17901340039176797
Fold 2 IBS: 0.20307707168836137
Fold 3 IBS: 0.17478517763412116
Fold 4 IBS: 0.14248300642108122
Fold 5 IBS: 0.26916639994419245
[I 2024-04-13 18:01:09,003] Trial 77 finished with value: 0.19370501121590483 and parameters: {'l1_ratio': 0.9356194012373432}. Best is trial 47 with value: 0.193689053

In [51]:
train_cindex['CoxElastic'] = np.round(study_cindex.best_value, 3)
train_ibs['CoxElastic'] = np.round(study_ibs.best_value, 3)

In [52]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.756
train_ibs:  0.194


#### Test

In [53]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [54]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    return model_class(**best_params, fit_baseline_model=True)

# Set the best model 
best_model_cindex = create_best_model(CoxnetSurvivalAnalysis, 
                                      study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex :", c_index)

# Set the best model 
best_model_ibs = create_best_model(CoxnetSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
gbsg_times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(gbsg_times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, gbsg_times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=0.050990548882846466)

test_cindex : 0.589


CoxnetSurvivalAnalysis(fit_baseline_model=True, l1_ratio=0.9992471693779538)

test_ibs:  0.232


In [55]:
# Saving the values to the dictionary 
test_cindex['CoxElastic'] = c_index
test_ibs['CoxElastic'] = ibs

### 5. Random Survival Forest

#### Train

In [56]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        oob_score = trial.suggest_categorical("oob_score", [True, False])
        max_samples = trial.suggest_float("max_samples", 0.1, 1.0) 
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None])
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        
        # Include warm_start for C-index optimization
        if metric == "c-index":
            warm_start = trial.suggest_categorical("warm_start", [True, False])
        else:
            warm_start = False  # Exclude warm_start for other metrics
        
        # Create and fit survival model 
        model = model_class(min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            max_leaf_nodes=max_leaf_nodes,
                            n_estimators=n_estimators, 
                            oob_score=oob_score,
                            warm_start=warm_start,
                            max_depth=max_depth,
                            max_features=max_features,
                            min_weight_fraction_leaf=min_weight_fraction_leaf, 
                            max_samples=max_samples, 
                            random_state=123)

        scores = [] 

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])
            
            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                gbsg_times = np.arange(lower, upper)
                cox_surv_prob = np.row_stack([fn(gbsg_times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, cox_surv_prob, gbsg_times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(RandomSurvivalForest, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(RandomSurvivalForest, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-13 18:01:58,356] A new study created in memory with name: no-name-add86d19-4f9f-48a1-8ff8-21a1fb86e7bd


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.7424242424242424
Fold 2 C-index: 0.7232142857142857
Fold 3 C-index: 0.8186274509803921
Fold 4 C-index: 0.8713080168776371
Fold 5 C-index: 0.6737089201877934
[I 2024-04-13 18:02:10,907] Trial 0 finished with value: 0.7658565832368702 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 7, 'min_samples_leaf': 5, 'max_depth': 12, 'n_estimators': 360, 'oob_score': False, 'max_samples': 0.7163467647263769, 'max_features': None, 'min_weight_fraction_leaf': 0.2192861223398122, 'warm_start': False}. Best is trial 0 with value: 0.7658565832368702.
Fold 1 C-index: 0.6688311688311688
Fold 2 C-index: 0.71875
Fold 3 C-index: 0.7058823529411765
Fold 4 C-index: 0.8016877637130801
Fold 5 C-index: 0.6995305164319249
[I 2024-04-13 18:02:21,273] Trial 1 finished with value: 0.7189363603834701 and parameters: {'min_samples_split': 16, 'max_leaf_nodes': 5, 'min_samples_leaf': 4, 'max_depth': 11, 'n_estimators': 266, 'oob_score': False, 'max_samples': 0.7520097923745717, 'max_featur

Fold 1 C-index: 0.6385281385281385
Fold 2 C-index: 0.7098214285714286
Fold 3 C-index: 0.7083333333333334
Fold 4 C-index: 0.8291139240506329
Fold 5 C-index: 0.6291079812206573
[I 2024-04-13 18:04:25,880] Trial 16 finished with value: 0.7029809611408382 and parameters: {'min_samples_split': 12, 'max_leaf_nodes': 2, 'min_samples_leaf': 5, 'max_depth': 20, 'n_estimators': 175, 'oob_score': False, 'max_samples': 0.5506605211937063, 'max_features': None, 'min_weight_fraction_leaf': 0.00014872775190403337, 'warm_start': False}. Best is trial 0 with value: 0.7658565832368702.
Fold 1 C-index: 0.6103896103896104
Fold 2 C-index: 0.7276785714285714
Fold 3 C-index: 0.6225490196078431
Fold 4 C-index: 0.689873417721519
Fold 5 C-index: 0.6314553990610329
[I 2024-04-13 18:04:30,089] Trial 17 finished with value: 0.6563892036417154 and parameters: {'min_samples_split': 17, 'max_leaf_nodes': 8, 'min_samples_leaf': 1, 'max_depth': 14, 'n_estimators': 131, 'oob_score': False, 'max_samples': 0.8459339712389

Fold 1 C-index: 0.7056277056277056
Fold 2 C-index: 0.734375
Fold 3 C-index: 0.8259803921568627
Fold 4 C-index: 0.8670886075949367
Fold 5 C-index: 0.687793427230047
[I 2024-04-13 18:06:23,804] Trial 31 finished with value: 0.7641730265219104 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 4, 'min_samples_leaf': 15, 'max_depth': 12, 'n_estimators': 250, 'oob_score': True, 'max_samples': 0.7987771976681688, 'max_features': None, 'min_weight_fraction_leaf': 0.21375919595874837, 'warm_start': True}. Best is trial 29 with value: 0.7685603760823929.
Fold 1 C-index: 0.6277056277056277
Fold 2 C-index: 0.8102678571428571
Fold 3 C-index: 0.6936274509803921
Fold 4 C-index: 0.8291139240506329
Fold 5 C-index: 0.7347417840375586
[I 2024-04-13 18:06:27,865] Trial 32 finished with value: 0.7390913287834138 and parameters: {'min_samples_split': 2, 'max_leaf_nodes': 4, 'min_samples_leaf': 15, 'max_depth': 12, 'n_estimators': 249, 'oob_score': True, 'max_samples': 0.7603792028308581, 'max_featu

...

Fold 5 IBS: 0.23185076637405122
[I 2024-04-13 18:19:29,404] Trial 31 finished with value: 0.1944706232042052 and parameters: {'min_samples_split': 10, 'max_leaf_nodes': 15, 'min_samples_leaf': 5, 'max_depth': 5, 'n_estimators': 248, 'oob_score': False, 'max_samples': 0.5943093092467338, 'max_features': 'auto', 'min_weight_fraction_leaf': 0.045073432551245296}. Best is trial 27 with value: 0.19393098799618166.
Fold 1 IBS: 0.19189320796178416
Fold 2 IBS: 0.2052610312293916
Fold 3 IBS: 0.1861107153615012
Fold 4 IBS: 0.16719140666147253
Fold 5 IBS: 0.22755309800616683
[I 2024-04-13 18:19:37,398] Trial 32 finished with value: 0.19560189184406326 and parameters: {'min_samples_split': 12, 'max_leaf_nodes': 14, 'min_samples_leaf': 4, 'max_depth': 4, 'n_estimators': 255, 'oob_score': False, 'max_samples': 0.6062923942533682, 'max_features': 'auto', 'min_weight_fraction_leaf': 0.09176370291993877}. Best is trial 27 with value: 0.19393098799618166.
Fold 1 IBS: 0.18633076489355724
Fold 2 IBS: 0.21

Fold 1 IBS: 0.18831038256552204
Fold 2 IBS: 0.20330959953611447
Fold 3 IBS: 0.1851297747224047
Fold 4 IBS: 0.17704850491733026
Fold 5 IBS: 0.21187395154278196
[I 2024-04-13 18:21:14,359] Trial 47 finished with value: 0.1931344426568307 and parameters: {'min_samples_split': 19, 'max_leaf_nodes': 8, 'min_samples_leaf': 1, 'max_depth': 1, 'n_estimators': 137, 'oob_score': True, 'max_samples': 0.893618602031134, 'max_features': 'auto', 'min_weight_fraction_leaf': 0.025121977917597738}. Best is trial 34 with value: 0.19091252378673845.
Fold 1 IBS: 0.22231357319417014
Fold 2 IBS: 0.21299690199828966
Fold 3 IBS: 0.2020657967661787
Fold 4 IBS: 0.21789212937822908
Fold 5 IBS: 0.2150793184508971
[I 2024-04-13 18:21:18,822] Trial 48 finished with value: 0.21406954395755293 and parameters: {'min_samples_split': 19, 'max_leaf_nodes': 5, 'min_samples_leaf': 8, 'max_depth': 1, 'n_estimators': 108, 'oob_score': True, 'max_samples': 0.9168117483275655, 'max_features': None, 'min_weight_fraction_leaf': 

Fold 1 IBS: 0.18940722653110123
Fold 2 IBS: 0.20195603211835453
Fold 3 IBS: 0.18524950719590635
Fold 4 IBS: 0.18167411231663144
Fold 5 IBS: 0.21301167725555065
[I 2024-04-13 18:22:28,890] Trial 63 finished with value: 0.19425971108350884 and parameters: {'min_samples_split': 18, 'max_leaf_nodes': 5, 'min_samples_leaf': 12, 'max_depth': 1, 'n_estimators': 187, 'oob_score': True, 'max_samples': 0.9709041364981504, 'max_features': 'auto', 'min_weight_fraction_leaf': 0.17457092877839805}. Best is trial 34 with value: 0.19091252378673845.
Fold 1 IBS: 0.19042077788605422
Fold 2 IBS: 0.20503437363791857
Fold 3 IBS: 0.1812573535917489
Fold 4 IBS: 0.1599480522032286
Fold 5 IBS: 0.22133756944780028
[I 2024-04-13 18:22:32,590] Trial 64 finished with value: 0.19159962535335012 and parameters: {'min_samples_split': 16, 'max_leaf_nodes': 3, 'min_samples_leaf': 9, 'max_depth': 11, 'n_estimators': 92, 'oob_score': True, 'max_samples': 0.9232561076825374, 'max_features': 'auto', 'min_weight_fraction_le

Fold 1 IBS: 0.18941186315145053
Fold 2 IBS: 0.20894591941191445
Fold 3 IBS: 0.187373667402488
Fold 4 IBS: 0.1572712742266229
Fold 5 IBS: 0.23558395033292584
[I 2024-04-13 18:23:42,147] Trial 79 finished with value: 0.19571733490508034 and parameters: {'min_samples_split': 17, 'max_leaf_nodes': 11, 'min_samples_leaf': 9, 'max_depth': 4, 'n_estimators': 66, 'oob_score': True, 'max_samples': 0.840513464959072, 'max_features': 'auto', 'min_weight_fraction_leaf': 0.017278165075527617}. Best is trial 34 with value: 0.19091252378673845.
Fold 1 IBS: 0.1874686284373422
Fold 2 IBS: 0.19665174781158837
Fold 3 IBS: 0.1871877356508405
Fold 4 IBS: 0.1763663924698438
Fold 5 IBS: 0.21229602140119605
[I 2024-04-13 18:23:46,080] Trial 80 finished with value: 0.1919941051541622 and parameters: {'min_samples_split': 19, 'max_leaf_nodes': 9, 'min_samples_leaf': 7, 'max_depth': 1, 'n_estimators': 91, 'oob_score': True, 'max_samples': 0.8671378958998438, 'max_features': 'auto', 'min_weight_fraction_leaf': 0.

Fold 1 IBS: 0.18624856075855828
Fold 2 IBS: 0.1984491501857183
Fold 3 IBS: 0.1837881604977131
Fold 4 IBS: 0.1616422821334367
Fold 5 IBS: 0.22637023753308694
[I 2024-04-13 18:24:39,151] Trial 95 finished with value: 0.19129967822170266 and parameters: {'min_samples_split': 17, 'max_leaf_nodes': 7, 'min_samples_leaf': 5, 'max_depth': 2, 'n_estimators': 41, 'oob_score': True, 'max_samples': 0.7371655958571915, 'max_features': 'auto', 'min_weight_fraction_leaf': 0.09280511653934975}. Best is trial 93 with value: 0.18721771570725532.
Fold 1 IBS: 0.18864761738406893
Fold 2 IBS: 0.20913810852224018
Fold 3 IBS: 0.1877503271050813
Fold 4 IBS: 0.15313275677333474
Fold 5 IBS: 0.23158574356531922
[I 2024-04-13 18:24:42,152] Trial 96 finished with value: 0.19405091067000887 and parameters: {'min_samples_split': 17, 'max_leaf_nodes': 8, 'min_samples_leaf': 4, 'max_depth': 3, 'n_estimators': 46, 'oob_score': True, 'max_samples': 0.7523032159371248, 'max_features': 'auto', 'min_weight_fraction_leaf': 

In [57]:
train_cindex['Randomsurvivalforest'] = np.round(study_cindex.best_value, 3)
train_ibs['Randomsurvivalforest'] = np.round(study_ibs.best_value, 3)

In [58]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.774
train_ibs:  0.187


#### Test

In [59]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))
    
y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [60]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(RandomSurvivalForest, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("test_cindex: ", c_index)

# Set the best model 
best_model_ibs = create_best_model(RandomSurvivalForest, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
gbsg_times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(gbsg_times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, gbsg_times)
ibs = np.round(ibs, 3)
print("test_ibs: ", ibs)

RandomSurvivalForest(max_depth=10, max_features=None, max_leaf_nodes=13,
                     max_samples=0.9157070212716846, min_samples_leaf=2,
                     min_samples_split=3,
                     min_weight_fraction_leaf=0.28086863940091983,
                     n_estimators=475, oob_score=True, random_state=123,
                     warm_start=True)

test_cindex:  0.669


RandomSurvivalForest(max_depth=2, max_features='auto', max_leaf_nodes=7,
                     max_samples=0.7957196161977251, min_samples_leaf=4,
                     min_samples_split=18,
                     min_weight_fraction_leaf=0.055989936415019606,
                     n_estimators=44, oob_score=True, random_state=123)

test_ibs:  0.21


In [61]:
# Saving the values to the dictionary 
test_cindex['Randomsurvivalforest'] = c_index
test_ibs['Randomsurvivalforest'] = ibs

### 6. ExtraSurvivalTrees

#### Train

In [62]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters 
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        oob_score = trial.suggest_categorical("oob_score", [True, False])
        warm_start = trial.suggest_categorical("warm_start", [True, False])
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None, 0.1, 1])
        max_samples = trial.suggest_float("max_samples", 0.1, 1.0) 
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        
        # Include warm_start for C-index optimization
        if metric == "c-index":
            warm_start = trial.suggest_categorical("warm_start", [True, False])
        else:
            warm_start = False  # Exclude warm_start for other metrics

        
        # Create and fit survival model 
        model = model_class(min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            max_leaf_nodes=max_leaf_nodes,
                            n_estimators=n_estimators, 
                            oob_score=oob_score, 
                            max_features=max_features, 
                            warm_start=warm_start, 
                            max_samples=max_samples,
                            min_weight_fraction_leaf=min_weight_fraction_leaf, 
                            max_depth=max_depth, 
                            random_state=123) 
        
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                gbsg_times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(gbsg_times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, gbsg_times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(ExtraSurvivalTrees, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(ExtraSurvivalTrees, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-13 18:24:54,115] A new study created in memory with name: no-name-d16e0299-c537-47d3-b70f-b37f9f96e965


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.7121212121212122
Fold 2 C-index: 0.7366071428571429
Fold 3 C-index: 0.8063725490196079
Fold 4 C-index: 0.8417721518987342
Fold 5 C-index: 0.6854460093896714
[I 2024-04-13 18:24:57,210] Trial 0 finished with value: 0.7564638130572738 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 7, 'min_samples_leaf': 5, 'max_depth': 12, 'n_estimators': 360, 'oob_score': False, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.7641958651588321, 'min_weight_fraction_leaf': 0.09124586522674999}. Best is trial 0 with value: 0.7564638130572738.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-13 18:25:04,796] Trial 1 finished with value: 0.5 and parameters: {'min_samples_split': 5, 'max_leaf_nodes': 12, 'min_samples_leaf': 11, 'max_depth': 13, 'n_estimators': 425, 'oob_score': True, 'warm_start': True, 'max_features': None, 'max_samples': 0.4877764869966794, 'min_weight_fraction_leaf': 0.2468425488251531

Fold 1 C-index: 0.7121212121212122
Fold 2 C-index: 0.7366071428571429
Fold 3 C-index: 0.803921568627451
Fold 4 C-index: 0.8417721518987342
Fold 5 C-index: 0.6596244131455399
[I 2024-04-13 18:26:26,984] Trial 16 finished with value: 0.750809297730016 and parameters: {'min_samples_split': 12, 'max_leaf_nodes': 10, 'min_samples_leaf': 7, 'max_depth': 20, 'n_estimators': 237, 'oob_score': False, 'warm_start': True, 'max_features': None, 'max_samples': 0.8988170687511211, 'min_weight_fraction_leaf': 0.1144909881867055}. Best is trial 0 with value: 0.7564638130572738.
Fold 1 C-index: 0.7121212121212122
Fold 2 C-index: 0.7299107142857143
Fold 3 C-index: 0.7892156862745098
Fold 4 C-index: 0.8417721518987342
Fold 5 C-index: 0.6596244131455399
[I 2024-04-13 18:26:29,326] Trial 17 finished with value: 0.746528835545142 and parameters: {'min_samples_split': 8, 'max_leaf_nodes': 15, 'min_samples_leaf': 13, 'max_depth': 9, 'n_estimators': 277, 'oob_score': False, 'warm_start': True, 'max_features': 

Fold 1 C-index: 0.7164502164502164
Fold 2 C-index: 0.7299107142857143
Fold 3 C-index: 0.803921568627451
Fold 4 C-index: 0.8628691983122363
Fold 5 C-index: 0.6596244131455399
[I 2024-04-13 18:27:07,701] Trial 31 finished with value: 0.7545552221642317 and parameters: {'min_samples_split': 18, 'max_leaf_nodes': 5, 'min_samples_leaf': 8, 'max_depth': 18, 'n_estimators': 78, 'oob_score': False, 'warm_start': True, 'max_features': None, 'max_samples': 0.870642154383752, 'min_weight_fraction_leaf': 0.07589392630375186}. Best is trial 22 with value: 0.767036894418702.
Fold 1 C-index: 0.7229437229437229
Fold 2 C-index: 0.7410714285714286
Fold 3 C-index: 0.8014705882352942
Fold 4 C-index: 0.8438818565400844
Fold 5 C-index: 0.6549295774647887
[I 2024-04-13 18:27:09,331] Trial 32 finished with value: 0.7528594347510638 and parameters: {'min_samples_split': 16, 'max_leaf_nodes': 3, 'min_samples_leaf': 9, 'max_depth': 19, 'n_estimators': 172, 'oob_score': False, 'warm_start': True, 'max_features': 

...

Fold 1 IBS: 0.18924477737479084
Fold 2 IBS: 0.21799018832154027
Fold 3 IBS: 0.19085735726014838
Fold 4 IBS: 0.19739387248975301
Fold 5 IBS: 0.22297685200412704
[I 2024-04-13 18:34:00,306] Trial 30 finished with value: 0.2036926094900719 and parameters: {'min_samples_split': 19, 'max_leaf_nodes': 19, 'min_samples_leaf': 6, 'max_depth': 11, 'n_estimators': 71, 'oob_score': False, 'warm_start': False, 'max_features': 'sqrt', 'max_samples': 0.8258632202024414, 'min_weight_fraction_leaf': 0.20241101766143602}. Best is trial 7 with value: 0.19566118486435183.
Fold 1 IBS: 0.1838565603858093
Fold 2 IBS: 0.21310814139547
Fold 3 IBS: 0.18816591366268318
Fold 4 IBS: 0.18856139746979875
Fold 5 IBS: 0.2241428176297601
[I 2024-04-13 18:34:04,159] Trial 31 finished with value: 0.19956696610870425 and parameters: {'min_samples_split': 15, 'max_leaf_nodes': 17, 'min_samples_leaf': 7, 'max_depth': 16, 'n_estimators': 133, 'oob_score': False, 'warm_start': False, 'max_features': 'sqrt', 'max_samples': 0.

Fold 1 IBS: 0.1703204366046531
Fold 2 IBS: 0.21025469102399516
Fold 3 IBS: 0.17796886722259822
Fold 4 IBS: 0.16660788588877593
Fold 5 IBS: 0.2275290168888809
[I 2024-04-13 18:35:52,288] Trial 45 finished with value: 0.19053617952578067 and parameters: {'min_samples_split': 10, 'max_leaf_nodes': 8, 'min_samples_leaf': 2, 'max_depth': 3, 'n_estimators': 299, 'oob_score': False, 'warm_start': True, 'max_features': 1, 'max_samples': 0.9986783002201525, 'min_weight_fraction_leaf': 0.004760507523990938}. Best is trial 45 with value: 0.19053617952578067.
Fold 1 IBS: 0.1716324348804869
Fold 2 IBS: 0.20993208788565448
Fold 3 IBS: 0.1789141654409001
Fold 4 IBS: 0.16837132365543245
Fold 5 IBS: 0.2264063005267728
[I 2024-04-13 18:36:03,076] Trial 46 finished with value: 0.19105126247784934 and parameters: {'min_samples_split': 8, 'max_leaf_nodes': 4, 'min_samples_leaf': 2, 'max_depth': 3, 'n_estimators': 286, 'oob_score': False, 'warm_start': True, 'max_features': 1, 'max_samples': 0.9907862185231

Fold 1 IBS: 0.184274939538523
Fold 2 IBS: 0.21449071346216647
Fold 3 IBS: 0.18683984328033068
Fold 4 IBS: 0.18862158723101846
Fold 5 IBS: 0.22342867248504114
[I 2024-04-13 18:38:24,017] Trial 60 finished with value: 0.19953115119941595 and parameters: {'min_samples_split': 6, 'max_leaf_nodes': 5, 'min_samples_leaf': 4, 'max_depth': 1, 'n_estimators': 264, 'oob_score': True, 'warm_start': True, 'max_features': 1, 'max_samples': 0.9208992051748564, 'min_weight_fraction_leaf': 0.03681620368970328}. Best is trial 53 with value: 0.18894626223842964.
Fold 1 IBS: 0.1836552976058727
Fold 2 IBS: 0.2136549386921458
Fold 3 IBS: 0.1867615542979982
Fold 4 IBS: 0.18959745987948876
Fold 5 IBS: 0.22360735647497929
[I 2024-04-13 18:38:35,540] Trial 61 finished with value: 0.19945532139009697 and parameters: {'min_samples_split': 8, 'max_leaf_nodes': 2, 'min_samples_leaf': 1, 'max_depth': 2, 'n_estimators': 292, 'oob_score': True, 'warm_start': True, 'max_features': 1, 'max_samples': 0.9964977138363789,

Fold 1 IBS: 0.17437167064723375
Fold 2 IBS: 0.2135778688097458
Fold 3 IBS: 0.18267487121953555
Fold 4 IBS: 0.17744152361550006
Fold 5 IBS: 0.22599792012746062
[I 2024-04-13 18:41:13,275] Trial 75 finished with value: 0.19481277088389515 and parameters: {'min_samples_split': 10, 'max_leaf_nodes': 6, 'min_samples_leaf': 2, 'max_depth': 4, 'n_estimators': 285, 'oob_score': True, 'warm_start': True, 'max_features': 'auto', 'max_samples': 0.8614518880476894, 'min_weight_fraction_leaf': 0.046111499155040925}. Best is trial 53 with value: 0.18894626223842964.
Fold 1 IBS: 0.18779799569555847
Fold 2 IBS: 0.2194030153528527
Fold 3 IBS: 0.19008139742953345
Fold 4 IBS: 0.1935917032465536
Fold 5 IBS: 0.22499236176871223
[I 2024-04-13 18:41:25,079] Trial 76 finished with value: 0.20317329469864212 and parameters: {'min_samples_split': 8, 'max_leaf_nodes': 2, 'min_samples_leaf': 1, 'max_depth': 1, 'n_estimators': 331, 'oob_score': True, 'warm_start': True, 'max_features': 'log2', 'max_samples': 0.902

Fold 1 IBS: 0.15808052483464857
Fold 2 IBS: 0.24513001845338123
Fold 3 IBS: 0.17353386158892606
Fold 4 IBS: 0.1414323854793232
Fold 5 IBS: 0.2630100664214536
[I 2024-04-13 18:44:03,680] Trial 90 finished with value: 0.19623737135554653 and parameters: {'min_samples_split': 6, 'max_leaf_nodes': 10, 'min_samples_leaf': 1, 'max_depth': 9, 'n_estimators': 344, 'oob_score': True, 'warm_start': True, 'max_features': None, 'max_samples': 0.8925947395705395, 'min_weight_fraction_leaf': 0.04616859454268613}. Best is trial 88 with value: 0.18836836078326372.
Fold 1 IBS: 0.16806680758384937
Fold 2 IBS: 0.23525934869183998
Fold 3 IBS: 0.17105625756502627
Fold 4 IBS: 0.13250701046302357
Fold 5 IBS: 0.24272974372536513
[I 2024-04-13 18:44:14,984] Trial 91 finished with value: 0.18992383360582088 and parameters: {'min_samples_split': 7, 'max_leaf_nodes': 10, 'min_samples_leaf': 1, 'max_depth': 8, 'n_estimators': 332, 'oob_score': True, 'warm_start': True, 'max_features': None, 'max_samples': 0.936503

In [63]:
train_cindex['ExtraSurvivalTrees'] = np.round(study_cindex.best_value, 3)
train_ibs['ExtraSurvivalTrees'] = np.round(study_ibs.best_value, 3)

In [64]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.787
train_ibs:  0.188


#### Test

In [65]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [66]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(ExtraSurvivalTrees, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(ExtraSurvivalTrees, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
gbsg_times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(gbsg_times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, gbsg_times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

ExtraSurvivalTrees(max_depth=19, max_features=None, max_leaf_nodes=16,
                   max_samples=0.9351618935202715, min_samples_leaf=1,
                   min_samples_split=15,
                   min_weight_fraction_leaf=0.035839978360771345,
                   n_estimators=132, random_state=123, warm_start=True)

C-index score: 0.673


ExtraSurvivalTrees(max_depth=8, max_features=1, max_leaf_nodes=10,
                   max_samples=0.8883209781210527, min_samples_leaf=1,
                   min_samples_split=7,
                   min_weight_fraction_leaf=0.0027749857774926326,
                   n_estimators=335, oob_score=True, random_state=123,
                   warm_start=True)

IBS: 0.203


In [67]:
# Saving the values to the dictionary 
test_cindex['ExtraSurvivalTrees'] = c_index
test_ibs['ExtraSurvivalTrees'] = ibs

### 7. GradientBoostingSurvivalAnalysis

#### Train

In [68]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

# Running to optuna for hyperparameter tuning
def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        subsample = trial.suggest_float("subsample", 0.1, 1)
        learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1)
        dropout_rate = trial.suggest_float("dropout_rate", 0.1, 1)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        criterion = trial.suggest_categorical('criterion', ['friedman_mse', 'squared_error'])
        ccp_alpha = trial.suggest_float("ccp_alpha", 0.0, 10)
        min_weight_fraction_leaf = trial.suggest_float("min_weight_fraction_leaf", 0.0, 0.5)
        max_features = trial.suggest_categorical("max_features", ["auto", "sqrt", "log2", None, 0.1, 1])
        min_impurity_decrease = trial.suggest_loguniform('min_impurity_decrease', 1e-7, 1e-1)
        validation_fraction = trial.suggest_float("validation_fraction", 0.0, 1.0)
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
        max_leaf_nodes = trial.suggest_int("max_leaf_nodes", 2, 20)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
        max_depth = trial.suggest_int("max_depth", 1, 20)
        
        # Create and fit survival model 
        model = model_class(subsample=subsample,
                            learning_rate=learning_rate,
                            dropout_rate=dropout_rate,
                            n_estimators=n_estimators,
                            ccp_alpha=ccp_alpha, 
                            criterion=criterion,
                            min_samples_split=min_samples_split,
                            min_samples_leaf=min_samples_leaf,
                            min_weight_fraction_leaf=min_weight_fraction_leaf,
                            max_depth=max_depth,
                            max_features=max_features,
                            max_leaf_nodes=max_leaf_nodes, 
                            min_impurity_decrease=min_impurity_decrease,
                            validation_fraction=validation_fraction, 
                            random_state=123)
        
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            model.fit(X_train, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                gbsg_times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(gbsg_times) for fn in model.predict_survival_function(X_test)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, gbsg_times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective

# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(GradientBoostingSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# Example usage for IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(GradientBoostingSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)

[I 2024-04-13 18:45:50,072] A new study created in memory with name: no-name-f659ab00-e558-4ae5-9d8c-249dcc3bb145


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-13 18:46:42,398] Trial 0 finished with value: 0.5 and parameters: {'subsample': 0.7268222670380755, 'learning_rate': 0.02932779416008757, 'dropout_rate': 0.3041663082077828, 'n_estimators': 276, 'criterion': 'friedman_mse', 'ccp_alpha': 9.807641983846155, 'min_weight_fraction_leaf': 0.34241486929243165, 'max_features': None, 'min_impurity_decrease': 2.4449249473284515e-05, 'validation_fraction': 0.7379954057320357, 'min_samples_split': 5, 'max_leaf_nodes': 5, 'min_samples_leaf': 11, 'max_depth': 11}. Best is trial 0 with value: 0.5.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-13 18:47:12,250] Trial 1 finished with value: 0.5 and parameters: {'subsample': 0.6709608626961889, 'learning_rate': 0.08509374761370117, 'dropout_rate': 0.7520097923745717, 'n_estimators': 306, 'criterion': 'friedman_mse', 'ccp_alpha': 3.

Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-13 19:02:12,747] Trial 13 finished with value: 0.5 and parameters: {'subsample': 0.8386796524426539, 'learning_rate': 0.046734492485875676, 'dropout_rate': 0.4821375662037144, 'n_estimators': 402, 'criterion': 'squared_error', 'ccp_alpha': 1.5696007313501796, 'min_weight_fraction_leaf': 0.18684147934268416, 'max_features': 'log2', 'min_impurity_decrease': 1.5044881127471587e-06, 'validation_fraction': 0.8166356053932342, 'min_samples_split': 16, 'max_leaf_nodes': 19, 'min_samples_leaf': 16, 'max_depth': 4}. Best is trial 9 with value: 0.7478429018287163.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.5
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-13 19:04:26,610] Trial 14 finished with value: 0.5 and parameters: {'subsample': 0.33235389014851724, 'learning_rate': 0.04522573411670834, 'dropout_rate': 0.2712811374536856, 'n_estimators': 405, 'criterion': 'square

Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5
[I 2024-04-13 19:27:28,272] Trial 25 finished with value: 0.5 and parameters: {'subsample': 0.7565257046780437, 'learning_rate': 0.01188344684416992, 'dropout_rate': 0.3433523077110169, 'n_estimators': 318, 'criterion': 'squared_error', 'ccp_alpha': 2.063559212730723, 'min_weight_fraction_leaf': 0.25401273903421573, 'max_features': 'auto', 'min_impurity_decrease': 5.773435946665558e-07, 'validation_fraction': 0.8062268184400477, 'min_samples_split': 16, 'max_leaf_nodes': 18, 'min_samples_leaf': 5, 'max_depth': 3}. Best is trial 9 with value: 0.7478429018287163.
Fold 1 C-index: 0.5
Fold 2 C-index: 0.6004464285714286
Fold 3 C-index: 0.5
Fold 4 C-index: 0.5
Fold 5 C-index: 0.5868544600938967
[I 2024-04-13 19:30:26,031] Trial 26 finished with value: 0.5374601777330652 and parameters: {'subsample': 0.8922404482621683, 'learning_rate': 0.011585260292674536, 'dropout_rate': 0.2527000999648632, 'n_estimators': 446, 'criterion': 'fried

...

Fold 3 IBS: 0.20453594732018135
Fold 4 IBS: 0.2247380413976268
Fold 5 IBS: 0.2181243152560957
[I 2024-04-13 21:47:01,753] Trial 44 finished with value: 0.21659054862241586 and parameters: {'subsample': 0.7438057966042957, 'learning_rate': 0.016732701733156254, 'dropout_rate': 0.28293775090891093, 'n_estimators': 476, 'criterion': 'squared_error', 'ccp_alpha': 1.6646055539220843, 'min_weight_fraction_leaf': 0.19668003262875977, 'max_features': 'auto', 'min_impurity_decrease': 2.6902494973410383e-07, 'validation_fraction': 0.8820166309308186, 'min_samples_split': 19, 'max_leaf_nodes': 19, 'min_samples_leaf': 9, 'max_depth': 12}. Best is trial 32 with value: 0.21569593361370903.
Fold 1 IBS: 0.21351546133478636
Fold 2 IBS: 0.2212953393166131
Fold 3 IBS: 0.20416066011138567
Fold 4 IBS: 0.22414784694013984
Fold 5 IBS: 0.21808198223903946
[I 2024-04-13 21:47:38,534] Trial 45 finished with value: 0.21624025798839286 and parameters: {'subsample': 0.8944196831300942, 'learning_rate': 0.007728654

Fold 3 IBS: 0.20453594732018138
Fold 4 IBS: 0.22473804139762676
Fold 5 IBS: 0.2181243152560957
[I 2024-04-13 22:08:20,636] Trial 55 finished with value: 0.21659054862241586 and parameters: {'subsample': 0.8826224600403735, 'learning_rate': 0.007074424640763303, 'dropout_rate': 0.13451729459411277, 'n_estimators': 150, 'criterion': 'squared_error', 'ccp_alpha': 1.3381569877935875, 'min_weight_fraction_leaf': 0.2467647950019329, 'max_features': None, 'min_impurity_decrease': 0.008146563872249941, 'validation_fraction': 0.8026232644824398, 'min_samples_split': 18, 'max_leaf_nodes': 18, 'min_samples_leaf': 14, 'max_depth': 16}. Best is trial 32 with value: 0.21569593361370903.
Fold 1 IBS: 0.21397652195149616
Fold 2 IBS: 0.22157791718667938
Fold 3 IBS: 0.20453594732018132
Fold 4 IBS: 0.22473804139762676
Fold 5 IBS: 0.21812431525609582
[I 2024-04-13 22:08:25,843] Trial 56 finished with value: 0.21659054862241592 and parameters: {'subsample': 0.28235337567058344, 'learning_rate': 0.0985092209

Fold 1 IBS: 0.21397652195149613
Fold 2 IBS: 0.22157791718667938
Fold 3 IBS: 0.20453594732018135
Fold 4 IBS: 0.2247380413976268
Fold 5 IBS: 0.2181243152560957
[I 2024-04-13 22:24:22,212] Trial 67 finished with value: 0.21659054862241586 and parameters: {'subsample': 0.7752653699906195, 'learning_rate': 0.013816164977086237, 'dropout_rate': 0.3423659929403322, 'n_estimators': 461, 'criterion': 'squared_error', 'ccp_alpha': 2.5489077007647944, 'min_weight_fraction_leaf': 0.07630547308149627, 'max_features': 'sqrt', 'min_impurity_decrease': 1.489839319569471e-07, 'validation_fraction': 0.8730218040479422, 'min_samples_split': 16, 'max_leaf_nodes': 18, 'min_samples_leaf': 18, 'max_depth': 2}. Best is trial 32 with value: 0.21569593361370903.
Fold 1 IBS: 0.21397652195149613
Fold 2 IBS: 0.22157791718667938
Fold 3 IBS: 0.20453594732018138
Fold 4 IBS: 0.2247380413976268
Fold 5 IBS: 0.21812431525609582
[I 2024-04-13 22:40:33,212] Trial 68 finished with value: 0.21659054862241592 and parameters: 

Fold 1 IBS: 0.21397652195149613
Fold 2 IBS: 0.22157791718667938
Fold 3 IBS: 0.20453594732018138
Fold 4 IBS: 0.2247380413976268
Fold 5 IBS: 0.2181243152560957
[I 2024-04-13 23:40:47,624] Trial 78 finished with value: 0.21659054862241586 and parameters: {'subsample': 0.8298863213670412, 'learning_rate': 0.00451364015632944, 'dropout_rate': 0.12600533500433567, 'n_estimators': 464, 'criterion': 'squared_error', 'ccp_alpha': 0.7257219477345627, 'min_weight_fraction_leaf': 0.13082025771859518, 'max_features': 'auto', 'min_impurity_decrease': 1.9076132907505136e-05, 'validation_fraction': 0.8664817216741038, 'min_samples_split': 19, 'max_leaf_nodes': 20, 'min_samples_leaf': 14, 'max_depth': 3}. Best is trial 32 with value: 0.21569593361370903.
Fold 1 IBS: 0.21397652195149613
Fold 2 IBS: 0.22157791718667935
Fold 3 IBS: 0.20453594732018138
Fold 4 IBS: 0.22473804139762676
Fold 5 IBS: 0.2181243152560957
[I 2024-04-13 23:42:14,819] Trial 79 finished with value: 0.21659054862241586 and parameters:

Fold 1 IBS: 0.21397652195149613
Fold 2 IBS: 0.22157791718667938
Fold 3 IBS: 0.20453594732018138
Fold 4 IBS: 0.22473804139762676
Fold 5 IBS: 0.21812431525609566
[I 2024-04-14 00:34:44,412] Trial 89 finished with value: 0.21659054862241586 and parameters: {'subsample': 0.8950685318650474, 'learning_rate': 0.0401442321523931, 'dropout_rate': 0.21800465633567007, 'n_estimators': 83, 'criterion': 'squared_error', 'ccp_alpha': 1.9753526447682455, 'min_weight_fraction_leaf': 0.12380998580246302, 'max_features': 'sqrt', 'min_impurity_decrease': 0.0005291280643839411, 'validation_fraction': 0.8620372359221313, 'min_samples_split': 19, 'max_leaf_nodes': 18, 'min_samples_leaf': 16, 'max_depth': 3}. Best is trial 32 with value: 0.21569593361370903.
Fold 1 IBS: 0.21397652195149613
Fold 2 IBS: 0.22157791718667938
Fold 3 IBS: 0.20453594732018138
Fold 4 IBS: 0.2247380413976268
Fold 5 IBS: 0.21812431525609582
[I 2024-04-14 00:35:03,949] Trial 90 finished with value: 0.21659054862241592 and parameters: 

In [69]:
train_cindex['GradientBoosting'] = np.round(study_cindex.best_value, 3)
train_ibs['GradientBoosting'] = np.round(study_ibs.best_value, 3)

In [70]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.748
train_ibs:  0.216


#### Test

In [71]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [72]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(GradientBoostingSurvivalAnalysis, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(GradientBoostingSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
gbsg_times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(gbsg_times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, gbsg_times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

GradientBoostingSurvivalAnalysis(ccp_alpha=0.07426378544613033,
                                 criterion='squared_error',
                                 dropout_rate=0.28125955124323376,
                                 learning_rate=0.013102111413618,
                                 max_features='auto', max_leaf_nodes=16,
                                 min_impurity_decrease=1.4994028685178666e-07,
                                 min_samples_leaf=10,
                                 min_weight_fraction_leaf=0.27579636299120275,
                                 n_estimators=406, random_state=123,
                                 subsample=0.6059965408578151,
                                 validation_fraction=0.6359003593513561)

C-index score: 0.676


GradientBoostingSurvivalAnalysis(ccp_alpha=0.01914282727239553,
                                 criterion='squared_error',
                                 dropout_rate=0.15717168328156264,
                                 learning_rate=0.008435334301058566,
                                 max_depth=2, max_features='auto',
                                 max_leaf_nodes=19,
                                 min_impurity_decrease=5.49537670642244e-07,
                                 min_samples_leaf=18, min_samples_split=19,
                                 min_weight_fraction_leaf=0.2499659379613939,
                                 n_estimators=472, random_state=123,
                                 subsample=0.8694330194615664,
                                 validation_fraction=0.9130609639429398)

IBS: 0.22


In [73]:
# Saving the values to the dictionary 
test_cindex['GradientBoosting'] = c_index
test_ibs['GradientBoosting'] = ibs

### 8. ComponentwiseGradientBoostingSurvivalAnalysis

#### Train

In [74]:
# Setting the y format 
y = clinical_train[['OS', 'event_OS']]

def create_objective(model_class, metric, X, y):
    def objective(trial): 
        # Suggest values for hyperparameters
        subsample = trial.suggest_float("subsample", 0.1, 1)
        dropout_rate = trial.suggest_float("dropout_rate", 0.1, 1)
        n_estimators = trial.suggest_int("n_estimators", 1, 500)
        learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1)
        
        # Create and fit survival model 
        model = model_class(subsample=subsample,
                            dropout_rate=dropout_rate,
                            n_estimators=n_estimators,
                            learning_rate=learning_rate,
                            random_state=123)
                
        scores = [] 
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
        
        for k, (train_index, test_index) in enumerate(skf.split(X, y.iloc[:, 1])): 
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train_df, y_test_df = y.iloc[train_index], y.iloc[test_index]
            
            # y_train into array 
            y_train = [] 
            for i, j in zip(y_train_df['event_OS'], y_train_df['OS']): 
                y_train.append((i, j))
            y_train = np.array(y_train, dtype=[('status', bool), ('time', np.int32)])

            # y_test into array
            y_test = [] 
            for i, j in zip(y_test_df['event_OS'], y_test_df['OS']): 
                y_test.append((i, j))
            y_test = np.array(y_test, dtype=[('status', bool), ('time', np.int32)])

            # Min Max Standardization
            excluded_columns = ['female', 
                                'cavum_oris',
                                'oropharynx',
                                'hypopharynx',
                                'larynx',
                                'histgrade_high',
                                'hpv_related',
                                'charlson',
                                'uicc8_III-IV'
                               ]
            excluded_columns = set(excluded_columns).intersection(X.columns)

            scaler = MinMaxScaler() 
            X_train_included = X_train.drop(excluded_columns, axis=1)
            X_test_included = X_test.drop(excluded_columns, axis=1)
                        
            if not X_train_included.empty and not X_test_included.empty:
                X_train_included_std = scaler.fit_transform(X_train_included)
                X_test_included_std = scaler.transform(X_test_included)
                
                # Concatenation
                X_train_std_df = pd.DataFrame(X_train_included_std, columns=X_train_included.columns, index=X_train_included.index)
                X_train_std = pd.concat([X_train_std_df, X_train[excluded_columns]], axis=1)

                X_test_std_df = pd.DataFrame(X_test_included_std, columns=X_test_included.columns, index=X_test_included.index)
                X_test_std = pd.concat([X_test_std_df, X_test[excluded_columns]], axis=1)
            
            else: 
                X_train_std = X_train
                X_test_std = X_test 
            
            model.fit(X_train_std, y_train)

            if metric == "c-index":
                # Make predictions using C-index 
                c_index_score = model.score(X_test_std, y_test)
                scores.append(c_index_score)
                print(f"Fold {k + 1} C-index: {c_index_score}")
                
            elif metric == "ibs":
                # Make predictions using IBS 
                lower, upper = np.percentile(y_test["time"], [10, 90])    
                gbsg_times = np.arange(lower, upper)
                surv_prob = np.row_stack([fn(gbsg_times) for fn in model.predict_survival_function(X_test_std)])
                ibs = integrated_brier_score(y_test, y_test, surv_prob, gbsg_times)
                scores.append(ibs)
                print(f"Fold {k + 1} IBS: {ibs}")
            else:
                raise ValueError("Invalid metric. Use 'C-index' or 'ibs'.")
        
        # Return the mean of scores
        return np.mean(scores)
    
    return objective


# C-index
study_cindex = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=123))
objective_cindex = create_objective(ComponentwiseGradientBoostingSurvivalAnalysis, "c-index", X_new, y)
study_cindex.optimize(objective_cindex, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for C-index: \n", study_cindex.best_trial)
print("\n")
print("* Best hyperparameters for C-index: \n", study_cindex.best_params)
print("\n")
print("* Best Score for C-index: \n", study_cindex.best_value)

# IBS
study_ibs = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=123))
objective_ibs = create_objective(ComponentwiseGradientBoostingSurvivalAnalysis, "ibs", X_new, y)
study_ibs.optimize(objective_ibs, n_trials=100, show_progress_bar=True)
print("\n")
print("* Best trial for IBS: \n", study_ibs.best_trial)
print("\n")
print("* Best hyperparameters for IBS: \n", study_ibs.best_params)
print("\n")
print("* Best Score for IBS: \n", study_ibs.best_value)


[I 2024-04-14 00:58:16,318] A new study created in memory with name: no-name-5e19b7e4-35ea-4f56-8b3d-fff945e3cf52


  0%|          | 0/100 [00:00<?, ?it/s]

Fold 1 C-index: 0.7056277056277056
Fold 2 C-index: 0.6004464285714286
Fold 3 C-index: 0.7328431372549019
Fold 4 C-index: 0.810126582278481
Fold 5 C-index: 0.5868544600938967
[I 2024-04-14 00:58:18,660] Trial 0 finished with value: 0.6871796627652828 and parameters: {'subsample': 0.7268222670380755, 'dropout_rate': 0.3575254014553415, 'n_estimators': 114, 'learning_rate': 0.05558016213920623}. Best is trial 0 with value: 0.6871796627652828.
Fold 1 C-index: 0.7012987012987013
Fold 2 C-index: 0.6004464285714286
Fold 3 C-index: 0.7328431372549019
Fold 4 C-index: 0.810126582278481
Fold 5 C-index: 0.5868544600938967
[I 2024-04-14 00:58:40,271] Trial 1 finished with value: 0.686313861899482 and parameters: {'subsample': 0.7475220728070068, 'dropout_rate': 0.4807958141120149, 'n_estimators': 491, 'learning_rate': 0.06879814411990147}. Best is trial 0 with value: 0.6871796627652828.
Fold 1 C-index: 0.7012987012987013
Fold 2 C-index: 0.6004464285714286
Fold 3 C-index: 0.7328431372549019
Fold 4 C

Fold 1 C-index: 0.6796536796536796
Fold 2 C-index: 0.7366071428571429
Fold 3 C-index: 0.7990196078431373
Fold 4 C-index: 0.8227848101265823
Fold 5 C-index: 0.676056338028169
[I 2024-04-14 01:01:44,410] Trial 19 finished with value: 0.7428243157017421 and parameters: {'subsample': 0.10013306718236009, 'dropout_rate': 0.9820749239416067, 'n_estimators': 431, 'learning_rate': 0.08300323114610605}. Best is trial 11 with value: 0.7455338965923975.
Fold 1 C-index: 0.6796536796536796
Fold 2 C-index: 0.7410714285714286
Fold 3 C-index: 0.803921568627451
Fold 4 C-index: 0.8227848101265823
Fold 5 C-index: 0.5868544600938967
[I 2024-04-14 01:01:52,464] Trial 20 finished with value: 0.7268571894146076 and parameters: {'subsample': 0.2630057481431337, 'dropout_rate': 0.5777838682216713, 'n_estimators': 258, 'learning_rate': 0.09795590448114745}. Best is trial 11 with value: 0.7455338965923975.
Fold 1 C-index: 0.6753246753246753
Fold 2 C-index: 0.7410714285714286
Fold 3 C-index: 0.7990196078431373
Fo

Fold 1 C-index: 0.70995670995671
Fold 2 C-index: 0.6004464285714286
Fold 3 C-index: 0.7328431372549019
Fold 4 C-index: 0.810126582278481
Fold 5 C-index: 0.5868544600938967
[I 2024-04-14 01:04:29,690] Trial 38 finished with value: 0.6880454636310838 and parameters: {'subsample': 0.5686484351119852, 'dropout_rate': 0.9273237630210112, 'n_estimators': 246, 'learning_rate': 0.07845082851951218}. Best is trial 11 with value: 0.7455338965923975.
Fold 1 C-index: 0.7575757575757576
Fold 2 C-index: 0.6004464285714286
Fold 3 C-index: 0.7328431372549019
Fold 4 C-index: 0.810126582278481
Fold 5 C-index: 0.5868544600938967
[I 2024-04-14 01:04:38,187] Trial 39 finished with value: 0.6975692731548933 and parameters: {'subsample': 0.8328204942141013, 'dropout_rate': 0.1771839477929863, 'n_estimators': 285, 'learning_rate': 0.013931445007643331}. Best is trial 11 with value: 0.7455338965923975.
Fold 1 C-index: 0.683982683982684
Fold 2 C-index: 0.7410714285714286
Fold 3 C-index: 0.803921568627451
Fold 4

...

Fold 2 IBS: 0.22098901295141482
Fold 3 IBS: 0.19045013870250915
Fold 4 IBS: 0.19457407998909504
Fold 5 IBS: 0.2297046752990242
[I 2024-04-14 01:12:17,710] Trial 19 finished with value: 0.20417214469524986 and parameters: {'subsample': 0.6203097276533311, 'dropout_rate': 0.7825648884185067, 'n_estimators': 169, 'learning_rate': 0.012702076480642453}. Best is trial 6 with value: 0.2014040617385498.
Fold 1 IBS: 0.1330635531817609
Fold 2 IBS: 0.2873945911931824
Fold 3 IBS: 0.20988152123608766
Fold 4 IBS: 0.13886518764316072
Fold 5 IBS: 0.3342176432895853
[I 2024-04-14 01:12:25,847] Trial 20 finished with value: 0.2206844993087554 and parameters: {'subsample': 0.38161734041406226, 'dropout_rate': 0.825887859456495, 'n_estimators': 315, 'learning_rate': 0.039831795316557894}. Best is trial 6 with value: 0.2014040617385498.
Fold 1 IBS: 0.19073287393187716
Fold 2 IBS: 0.22033336815614252
Fold 3 IBS: 0.1923689926601459
Fold 4 IBS: 0.19934903879334157
Fold 5 IBS: 0.22693510103337397
[I 2024-04-1

Fold 4 IBS: 0.18963358127091137
Fold 5 IBS: 0.23395808174187974
[I 2024-04-14 01:13:24,935] Trial 38 finished with value: 0.20300927417970324 and parameters: {'subsample': 0.9229861705051505, 'dropout_rate': 0.5445128971839069, 'n_estimators': 32, 'learning_rate': 0.07994143832657483}. Best is trial 31 with value: 0.20112673449582133.
Fold 1 IBS: 0.14089593072095502
Fold 2 IBS: 0.26511117850380844
Fold 3 IBS: 0.19677652632752524
Fold 4 IBS: 0.14729454625577706
Fold 5 IBS: 0.3047581909661958
[I 2024-04-14 01:13:28,680] Trial 39 finished with value: 0.2109672745548523 and parameters: {'subsample': 0.6847274160309886, 'dropout_rate': 0.6026791333381789, 'n_estimators': 182, 'learning_rate': 0.047260604312967414}. Best is trial 31 with value: 0.20112673449582133.
Fold 1 IBS: 0.13971317742997533
Fold 2 IBS: 0.2680025852392165
Fold 3 IBS: 0.19568605187963634
Fold 4 IBS: 0.14626529602655097
Fold 5 IBS: 0.30607863999868745
[I 2024-04-14 01:13:31,444] Trial 40 finished with value: 0.21114915011

Fold 4 IBS: 0.1421765996020749
Fold 5 IBS: 0.3198842216652
[I 2024-04-14 01:14:15,476] Trial 57 finished with value: 0.22005074316833056 and parameters: {'subsample': 0.13165605060453595, 'dropout_rate': 0.31377442599984917, 'n_estimators': 151, 'learning_rate': 0.07355231920702918}. Best is trial 48 with value: 0.2007926295608577.
Fold 1 IBS: 0.16761507596154626
Fold 2 IBS: 0.22803850923004326
Fold 3 IBS: 0.18654619322633878
Fold 4 IBS: 0.17717309570614478
Fold 5 IBS: 0.24622975248472237
[I 2024-04-14 01:14:17,738] Trial 58 finished with value: 0.2011205253217591 and parameters: {'subsample': 0.7705771626423796, 'dropout_rate': 0.7021812895625077, 'n_estimators': 122, 'learning_rate': 0.030622252622745642}. Best is trial 48 with value: 0.2007926295608577.
Fold 1 IBS: 0.18874788007962418
Fold 2 IBS: 0.2204792981603036
Fold 3 IBS: 0.19149147767111682
Fold 4 IBS: 0.19700150139666606
Fold 5 IBS: 0.22773666772180465
[I 2024-04-14 01:14:19,978] Trial 59 finished with value: 0.20509136500590

Fold 5 IBS: 0.23080668782285596
[I 2024-04-14 01:14:50,020] Trial 76 finished with value: 0.20405445875275702 and parameters: {'subsample': 0.729382594130491, 'dropout_rate': 0.6783619617211385, 'n_estimators': 72, 'learning_rate': 0.03086841519700664}. Best is trial 48 with value: 0.2007926295608577.
Fold 1 IBS: 0.16539769706536414
Fold 2 IBS: 0.22990940170380345
Fold 3 IBS: 0.18631811547303107
Fold 4 IBS: 0.17398371273466193
Fold 5 IBS: 0.25027000215867606
[I 2024-04-14 01:14:53,233] Trial 77 finished with value: 0.20117578582710735 and parameters: {'subsample': 0.8562487404883022, 'dropout_rate': 0.5355841488171579, 'n_estimators': 158, 'learning_rate': 0.025512570062193523}. Best is trial 48 with value: 0.2007926295608577.
Fold 1 IBS: 0.1861460260765977
Fold 2 IBS: 0.2207349422884218
Fold 3 IBS: 0.19116024662320327
Fold 4 IBS: 0.19638098197841433
Fold 5 IBS: 0.22916888150998138
[I 2024-04-14 01:14:55,139] Trial 78 finished with value: 0.20471821569532372 and parameters: {'subsample

Fold 5 IBS: 0.2530438768553521
[I 2024-04-14 01:15:55,402] Trial 95 finished with value: 0.2011261260458114 and parameters: {'subsample': 0.8422246435924808, 'dropout_rate': 0.5964014242242069, 'n_estimators': 127, 'learning_rate': 0.03356108802245582}. Best is trial 48 with value: 0.2007926295608577.
Fold 1 IBS: 0.16833703881776008
Fold 2 IBS: 0.2272152857140041
Fold 3 IBS: 0.1866950376112
Fold 4 IBS: 0.17788130795097382
Fold 5 IBS: 0.24530983172104343
[I 2024-04-14 01:15:57,508] Trial 96 finished with value: 0.20108770036299628 and parameters: {'subsample': 0.8205106604823191, 'dropout_rate': 0.6501086479099357, 'n_estimators': 109, 'learning_rate': 0.03318174004435172}. Best is trial 48 with value: 0.2007926295608577.
Fold 1 IBS: 0.16600966254812746
Fold 2 IBS: 0.22917548167323062
Fold 3 IBS: 0.1863906804089131
Fold 4 IBS: 0.17530834913167373
Fold 5 IBS: 0.24859220802093854
[I 2024-04-14 01:15:59,806] Trial 97 finished with value: 0.20109527635657667 and parameters: {'subsample': 0.

In [75]:
train_cindex['ComponentwiseGradientBoosting'] = np.round(study_cindex.best_value, 3)
train_ibs['ComponentwiseGradientBoosting'] = np.round(study_ibs.best_value, 3)

In [76]:
print("train_cindex: ", np.round(study_cindex.best_value, 3))
print("train_ibs: ", np.round(study_ibs.best_value, 3))

train_cindex:  0.751
train_ibs:  0.201


#### Test

In [77]:
# y into array 
lists = [] 
for i, j in zip(y['event_OS'], y['OS']): 
    lists.append((i, j))

y = np.array(lists, dtype=[('status', bool), ('time', np.int32)])

In [78]:
# A function for building the best model with the best parameters 
def create_best_model(model_class, best_params):
    best_params["random_state"]=123
    return model_class(**best_params)

# Set the best model 
best_model_cindex = create_best_model(ComponentwiseGradientBoostingSurvivalAnalysis, study_cindex.best_params)

# Train the best model for C-index on the whole dataset
best_model_cindex.fit(X_new_std, y)

# Evaluate the best model for C-index on MAASTRO dataset
c_index = best_model_cindex.score(MAASTRO_new_std, y_MAASTRO)
c_index = np.round(c_index, 3)
print("C-index score:", c_index)

# Set the best model 
best_model_ibs = create_best_model(ComponentwiseGradientBoostingSurvivalAnalysis, study_ibs.best_params)

# Train the best model for IBS on the whole dataset
best_model_ibs.fit(X_new_std, y)

# Evaluate the best model for IBS on MAASTRO dataset
lower, upper = np.percentile(y_MAASTRO["time"], [10, 90])
gbsg_times = np.arange(lower, upper)
surv_prob = np.row_stack([fn(gbsg_times) for fn in best_model_ibs.predict_survival_function(MAASTRO_new_std)])
ibs = integrated_brier_score(y_MAASTRO, y_MAASTRO, surv_prob, gbsg_times)
ibs = np.round(ibs, 3)
print("IBS:", ibs)

ComponentwiseGradientBoostingSurvivalAnalysis(dropout_rate=0.6672299985480628,
                                              learning_rate=0.03906201007616112,
                                              n_estimators=225,
                                              random_state=123,
                                              subsample=0.3385180226064827)

C-index score: 0.663


ComponentwiseGradientBoostingSurvivalAnalysis(dropout_rate=0.5685805772015377,
                                              learning_rate=0.07550425969628831,
                                              n_estimators=55, random_state=123,
                                              subsample=0.6626333762174762)

IBS: 0.207


In [79]:
# Saving the values to the dictionary 
test_cindex['ComponentwiseGradientBoosting'] = c_index
test_ibs['ComponentwiseGradientBoosting'] = ibs

## Results

In [80]:
df_train_cindex = pd.DataFrame(train_cindex, index=['C-index']).transpose().sort_values(by='C-index', ascending=False)
df_train_cindex['rank'] = df_train_cindex['C-index'].rank(ascending=False)
df_train_cindex 

,C-index,rank
ExtraSurvivalTrees,0.787,1.0
Randomsurvivalforest,0.774,2.0
CoxElastic,0.756,3.0
ComponentwiseGradientBoosting,0.751,4.0
GradientBoosting,0.748,5.0
CoxPH,0.736,6.5
CoxLasso,0.736,6.5
CoxRidge,0.698,8.0


In [81]:
df_train_ibs = pd.DataFrame(train_ibs, index=['IBS']).transpose().sort_values(by='IBS', ascending=True)
df_train_ibs['rank'] = df_train_ibs['IBS'].rank(ascending=True)
df_train_ibs

,IBS,rank
Randomsurvivalforest,0.187,1.0
ExtraSurvivalTrees,0.188,2.0
CoxPH,0.194,4.0
CoxLasso,0.194,4.0
CoxElastic,0.194,4.0
ComponentwiseGradientBoosting,0.201,6.0
GradientBoosting,0.216,7.0
CoxRidge,0.217,8.0


In [82]:
df_test_cindex = pd.DataFrame(test_cindex, index=['C-index']).transpose().sort_values(by='C-index', ascending=False)
df_test_cindex['rank'] = df_test_cindex['C-index'].rank(ascending=False)
df_test_cindex 

,C-index,rank
GradientBoosting,0.676,1.0
ExtraSurvivalTrees,0.673,2.0
Randomsurvivalforest,0.669,3.0
ComponentwiseGradientBoosting,0.663,4.0
CoxLasso,0.624,5.0
CoxPH,0.623,6.0
CoxRidge,0.589,7.5
CoxElastic,0.589,7.5


In [83]:
df_test_ibs = pd.DataFrame(test_ibs, index=['IBS']).transpose().sort_values(by='IBS', ascending=True)
df_test_ibs['rank'] = df_test_ibs['IBS'].rank(ascending=True)
df_test_ibs

,IBS,rank
ExtraSurvivalTrees,0.203,1.0
ComponentwiseGradientBoosting,0.207,2.0
Randomsurvivalforest,0.210,3.0
GradientBoosting,0.220,4.0
CoxRidge,0.221,5.0
CoxLasso,0.232,6.5
CoxElastic,0.232,6.5
CoxPH,0.234,8.0


In [84]:
# Renaming the column "index" to "model" 
df_train_cindex = df_train_cindex.reset_index().rename(columns={"index": "model"})
df_train_ibs = df_train_ibs.reset_index().rename(columns={"index": "model"})
df_test_cindex = df_test_cindex.reset_index().rename(columns={"index": "model"})
df_test_ibs = df_test_ibs.reset_index().rename(columns={"index": "model"})

# Save the files 
dfs = [df_train_cindex, df_train_ibs, df_test_cindex, df_test_ibs]  
file_path = 'path_to_your_folder/'  

# List of corresponding file names
file_names = ['train_cindex.csv', 'train_ibs.csv', 'test_cindex.csv', 'test_ibs.csv']


dfs = [df_train_cindex, df_train_ibs, df_test_cindex, df_test_ibs]  
file_path = '/Users/minjeongcheon/Desktop/results_thesis/d1/os/minmax/plsr/'  

# List of corresponding file names
file_names = ['train_cindex.csv', 'train_ibs.csv', 'test_cindex.csv', 'test_ibs.csv']

# Modify the file names to match the desired format
modified_file_names = ['d1_os_minmax_plsr_' + file_name for file_name in file_names]

# Loop through each DataFrame and save them with corresponding modified file names
for df, modified_file_name in zip(dfs, modified_file_names):
    file_path_name = file_path + modified_file_name  # Construct the full file path
    df.to_csv(file_path_name, index=False)  # Save the DataFrame to CSV file


In [85]:
from datetime import date
today = date.today()
print("Date: ", today)

Date:  2024-04-14
